In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import Dropdown, interactive_output, display
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

# Загрузка данных (относительный путь)
df = pd.read_excel('reparsed/data.xlsx', header=1)

# Список заболеваний
diseases = ['АГ', 'ИБС', 'ЦВЗ', 'ССС', 'СД', 'ЗНО', 'ХОБЛ-БА', 'ЯБЖДК']
disease_names_ru = {
    'АГ': 'Гипертония',
    'ИБС': 'ИБС',
    'ЦВЗ': 'Цереброваскулярные',
    'ССС': 'ССС (другие)',
    'СД': 'Сахарный диабет',
    'ЗНО': 'Онкология',
    'ХОБЛ-БА': 'ХОБЛ/Астма',
    'ЯБЖДК': 'Язвенная болезнь'
}

# Агрегация по регионам
region_data = df.groupby('РЕГИОН').agg({
    '1 этап': 'sum',
    '2 этап': 'sum',
    **{d: 'sum' for d in diseases}
}).reset_index()

# Нормировка на 1000 человек
for d in diseases:
    region_data[f'{d}_per1000'] = region_data[d] / region_data['1 этап'] * 1000

# Список регионов
all_regions = sorted(region_data['РЕГИОН'].unique())
regions_with_all = ['Все регионы'] + all_regions

def plot_dashboard(selected_region='Все регионы'):
    if selected_region == 'Все регионы':
        row = {
            '1 этап': region_data['1 этап'].sum(),
            '2 этап': region_data['2 этап'].sum(),
        }
        for d in diseases:
            row[d] = region_data[d].sum()
            row[f'{d}_per1000'] = row[d] / row['1 этап'] * 1000
        title_prefix = "ВСЕ РЕГИОНЫ (суммарно)"
        is_aggregated = True
    else:
        row = region_data[region_data['РЕГИОН'] == selected_region].iloc[0].to_dict()
        title_prefix = selected_region
        is_aggregated = False
    
    fig = plt.figure(figsize=(16, 12))
    
    # 1. ЭТАПНОСТЬ
    ax1 = plt.subplot(2, 2, 1)
    stages = [row['1 этап'], row['2 этап']]
    bars1 = ax1.bar(['1 этап', '2 этап'], stages, width=0.5, color=['#2E86AB', '#A23B72'], edgecolor='black', linewidth=1)
    ax1.set_ylabel('Количество человек', fontsize=11)
    ax1.set_title(f'{title_prefix}\n1 и 2 этапы диспансеризации', fontsize=12, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars1, stages):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stages)*0.01,
                f'{int(val):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 2. ВЫЯВЛЯЕМОСТЬ (вертикальные столбцы)
    ax2 = plt.subplot(2, 2, 2)
    disease_vals = [row[f'{d}_per1000'] for d in diseases]
    disease_labels = [disease_names_ru[d] for d in diseases]
    bars2 = ax2.bar(disease_labels, disease_vals, width=0.6, color='#2E86AB', edgecolor='black', linewidth=1)
    ax2.set_ylabel('Случаев на 1000 человек', fontsize=11)
    ax2.set_title(f'{title_prefix}\nВыявляемость заболеваний', fontsize=12, fontweight='bold')
    ax2.tick_params(axis='x', rotation=45, labelsize=9)
    ax2.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars2, disease_vals):
        if val > 0:
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(disease_vals)*0.01,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=8)
    
    # 3. СТРУКТУРА ЗАБОЛЕВАЕМОСТИ (доли)
    ax3 = plt.subplot(2, 2, 3)
    disease_abs = [row[d] for d in diseases]
    total = sum(disease_abs)
    percentages = [d/total*100 for d in disease_abs]
    sorted_idx = sorted(range(len(percentages)), key=lambda i: percentages[i])
    percentages_sorted = [percentages[i] for i in sorted_idx]
    labels_sorted = [disease_labels[i] for i in sorted_idx]
    colors = sns.color_palette("viridis", len(diseases))
    colors_sorted = [colors[i] for i in sorted_idx]
    bars3 = ax3.barh(labels_sorted, percentages_sorted, height=0.6, color=colors_sorted, edgecolor='black', linewidth=1)
    ax3.set_xlabel('Доля от всех заболеваний (%)', fontsize=11)
    ax3.set_title(f'{title_prefix}\nСтруктура заболеваемости', fontsize=12, fontweight='bold')
    for bar, val in zip(bars3, percentages_sorted):
        if val > 1:
            ax3.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%', va='center', fontsize=9)
    
    # 4. ТОП-15 РЕГИОНОВ
    ax4 = plt.subplot(2, 2, 4)
    if not is_aggregated:
        top15 = region_data.set_index('РЕГИОН')['АГ_per1000'].sort_values(ascending=False).head(15)
        rank = list(top15.index).index(selected_region) + 1 if selected_region in top15.index else None
        colors_bar = ['#A23B72' if i == selected_region else '#2E86AB' for i in top15.index]
        bars4 = ax4.barh(range(len(top15)), top15.values, color=colors_bar, height=0.7, edgecolor='black', linewidth=0.5)
        ax4.set_yticks(range(len(top15)))
        ax4.set_yticklabels(top15.index, fontsize=8)
        ax4.set_xlabel('Случаев АГ на 1000 человек', fontsize=11)
        title4 = f'ТОП-15 по АГ\n{selected_region} на {rank}-м месте' if rank else f'ТОП-15 по АГ\n{selected_region} не в топе'
        ax4.set_title(title4, fontsize=11, fontweight='bold')
        ax4.invert_yaxis()
        ax4.grid(axis='x', alpha=0.3)
    else:
        top15 = region_data.set_index('РЕГИОН')['АГ_per1000'].sort_values(ascending=False).head(15)
        bars4 = ax4.barh(range(len(top15)), top15.values, color='#2E86AB', height=0.7, edgecolor='black', linewidth=0.5)
        ax4.set_yticks(range(len(top15)))
        ax4.set_yticklabels(top15.index, fontsize=8)
        ax4.set_xlabel('Случаев АГ на 1000 человек', fontsize=11)
        ax4.set_title('ТОП-15 регионов по выявляемости АГ', fontsize=11, fontweight='bold')
        ax4.invert_yaxis()
        ax4.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Вывод статистики
    print("\n" + "="*70)
    print(f"СТАТИСТИКА ДЛЯ: {title_prefix}")
    print("="*70)
    print(f"1 этап: {row['1 этап']:,.0f} чел.")
    print(f"2 этап: {row['2 этап']:,.0f} чел. ({row['2 этап']/row['1 этап']*100:.1f}%)")
    print("\nВыявляемость на 1000 человек:")
    for d in diseases:
        print(f"  {disease_names_ru[d]}: {row[f'{d}_per1000']:.1f}")
    print("="*70)

# Создание выпадающего списка
region_dropdown = Dropdown(
    options=regions_with_all,
    value='Все регионы',
    description='Выберите регион:',
    style={'description_width': 'initial'},
    layout={'width': '350px'}
)

# Отображение виджета и графиков
display(region_dropdown)
out = interactive_output(plot_dashboard, {'selected_region': region_dropdown})
display(out)